In [ ]:
# In[1]:

import pandas as pd
import pytz

# timezone UTC+8
tz = pytz.timezone('Asia/Shanghai')

# Load files (reusing variable names where sensible)
df_metric = pd.read_csv('metric.csv')
df_log = pd.read_csv('log.csv')
df_trace = pd.read_csv('trace.csv')
df_err = pd.read_csv('error_logs.csv')

# Ensure timestamp columns are numeric and consistent
for df in (df_metric, df_log, df_trace, df_err):
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce')

# Helper to create compact timestamp summary (unix + localized ISO)
def ts_summary(s):
    # always return two tuples: (min_unix, min_local_str), (max_unix, max_local_str)
    if s is None or s.empty or s.dropna().empty:
        return (None, None), (None, None)
    mn = int(s.min())
    mx = int(s.max())
    mn_dt = pd.to_datetime(mn, unit='s', utc=True).tz_convert(tz)
    mx_dt = pd.to_datetime(mx, unit='s', utc=True).tz_convert(tz)
    return (mn, mn_dt.strftime('%Y-%m-%d %H:%M:%S %Z')), (mx, mx_dt.strftime('%Y-%m-%d %H:%M:%S %Z'))

# Metric file summary
metric_row_count = df_metric.shape[0]
metric_top_cmdb = df_metric['cmdb_id'].value_counts().head(20).rename_axis('cmdb_id').reset_index(name='count')
metric_kpi_counts = df_metric['kpi_name'].value_counts().head(20).rename_axis('kpi_name').reset_index(name='count')
(metric_min, metric_min_local), (metric_max, metric_max_local) = ts_summary(df_metric.get('timestamp', pd.Series(dtype='float')))
metric_sample = df_metric.head(5)

metric_report = {
    'row_count': metric_row_count,
    'unique_cmdb_top20': metric_top_cmdb,
    'unique_kpi_top20': metric_kpi_counts,
    'min_timestamp_unix_and_local': (metric_min, metric_min_local),
    'max_timestamp_unix_and_local': (metric_max, metric_max_local),
    'sample_rows_up_to_5': metric_sample
}

# Log file summary
log_row_count = df_log.shape[0]
log_top_cmdb = df_log['cmdb_id'].value_counts().head(20).rename_axis('cmdb_id').reset_index(name='count')
log_name_counts = df_log['log_name'].value_counts().head(20).rename_axis('log_name').reset_index(name='count')
(log_min, log_min_local), (log_max, log_max_local) = ts_summary(df_log.get('timestamp', pd.Series(dtype='float')))
log_sample = df_log.head(5)

log_report = {
    'row_count': log_row_count,
    'unique_cmdb_top20': log_top_cmdb,
    'unique_log_name_top20': log_name_counts,
    'min_timestamp_unix_and_local': (log_min, log_min_local),
    'max_timestamp_unix_and_local': (log_max, log_max_local),
    'sample_rows_up_to_5': log_sample
}

# Trace file summary
trace_row_count = df_trace.shape[0]
trace_top_cmdb = df_trace['cmdb_id'].value_counts().head(20).rename_axis('cmdb_id').reset_index(name='count')
trace_name_counts = df_trace['trace_name'].value_counts().head(20).rename_axis('trace_name').reset_index(name='count')
(trace_min, trace_min_local), (trace_max, trace_max_local) = ts_summary(df_trace.get('timestamp', pd.Series(dtype='float')))
trace_sample = df_trace.head(5)

trace_report = {
    'row_count': trace_row_count,
    'unique_cmdb_top20': trace_top_cmdb,
    'unique_trace_name_top20': trace_name_counts,
    'min_timestamp_unix_and_local': (trace_min, trace_min_local),
    'max_timestamp_unix_and_local': (trace_max, trace_max_local),
    'sample_rows_up_to_5': trace_sample
}

# Error logs file summary (no KPI column)
err_row_count = df_err.shape[0]
err_top_cmdb = df_err['cmdb_id'].value_counts().head(20).rename_axis('cmdb_id').reset_index(name='count')
# protect against missing message column
if 'message' in df_err.columns:
    err_message_counts = df_err['message'].value_counts().head(20).rename_axis('message').reset_index(name='count')
else:
    err_message_counts = pd.DataFrame(columns=['message', 'count'])
(err_min, err_min_local), (err_max, err_max_local) = ts_summary(df_err.get('timestamp', pd.Series(dtype='float')))
err_sample = df_err.head(5)

error_logs_report = {
    'row_count': err_row_count,
    'unique_cmdb_top20': err_top_cmdb,
    'top_messages_top20': err_message_counts,
    'min_timestamp_unix_and_local': (err_min, err_min_local),
    'max_timestamp_unix_and_local': (err_max, err_max_local),
    'sample_rows_up_to_5': err_sample
}

# Return compact reports (IPython will display them)
metric_report, log_report, trace_report, error_logs_report

```
Out[1]:
```
```python
summary = (
    "Summary of loaded telemetry files:\n\n"
    "metric.csv:\n"
    "- Total rows: 22,340\n"
    "- Top cmdb_id counts (top entries): Mysql01 3746, Mysql02 3715, Tomcat03 2061, "
    "Tomcat02 2034, Tomcat04 2016, Tomcat01 1953, MG02 1618, MG01 1609, Redis02 1151, IG01 333\n"
    "- Top kpi_name occurrences (top entries): app.sr 341, app.rr 341, app.mrt 341, app.cnt 341, "
    "OSLinux MEMORY/CPU related KPIs (several entries between ~210–279)\n"
    "- Timestamp range: min 1616481000 (2021-03-23 14:30:00 CST), max 1616482800 (2021-03-23 15:00:00 CST)\n"
    "- Sample (first rows): IG01 JVM memory and JVM CPU load metrics at timestamp 1616481000\n\n"
    "log.csv:\n"
    "- Total rows: 471\n"
    "- Top cmdb_id counts: Tomcat04 87, Tomcat02 86, Tomcat01 86, Tomcat03 86, apache01 62, apache02 62\n"
    "- Top log_name counts: log.error_count 187, log.row_count 187, log.gc_pause_max 97\n"
    "- Timestamp range: min 1616481000 (2021-03-23 14:30:00 CST), max 1616482800 (2021-03-23 15:00:00 CST)\n"
    "- Sample (first rows): Tomcat01 entries such as log.error_count=0.00, log.gc_pause_max=0.06, log.row_count=519 at 1616481000\n\n"
    "trace.csv:\n"
    "- Total rows: 264\n"
    "- Top cmdb_id counts: MG02 42, MG01 42, IG02 18, IG01 18, Tomcat01/02/03/04 each 18, several docker* hosts 18\n"
    "- Top trace_name occurrences (top entries): trace.from_root.duration_mean 12, trace.from_root.duration_p95 12, "
    "trace.from_root.row_count 12, trace.self.duration_mean 12, trace.self.duration_p95 12, trace.self.row_count 12\n"
    "- Timestamp range (both min and max are the same small value): 1616460 which corresponds to 1970-01-20 01:01:00 CST — "
    "this suggests trace timestamps differ from the others or are mis-scaled/formatted\n"
    "- Sample (first rows): IG01 trace.from_root and trace.self duration metrics with timestamp 1616460\n\n"
    "error_logs.csv:\n"
    "- Total rows: 0 (file is empty)\n"
    "- No cmdb_id, message, timestamp data present\n\n"
    "Notes:\n"
    "- metric and log files share a consistent timestamp window (2021-03-23 14:30 to 15:00 CST).\n"
    "- trace timestamps appear anomalous (very small Unix value) and should be checked/normalized before time-based correlation.\n"
    "- error_logs.csv contains no rows.\n"
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

({'row_count': 22340, 'unique_cmdb_top20':           cmdb_id  count
0         Mysql01   3746
1         Mysql02   3715
2        Tomcat03   2061
3        Tomcat02   2034
4        Tomcat04   2016
5        Tomcat01   1953
6            MG02   1618
7            MG01   1609
8         Redis02   1151
9            IG01    333
10           IG02    325
11       apache01    158
12        Redis01    132
13       apache02    125
14   ServiceTest1    124
15  ServiceTest10    124
16  ServiceTest11    124
17   ServiceTest7    124
18   ServiceTest6    124
19   ServiceTest5    124, 'unique_kpi_top20':                                              kpi_name  count
0                                              app.sr    341
1                                              app.rr    341
2                                             app.mrt    341
3                                             app.cnt    341
4        OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc    279
5                          OSLinux-CPU_CPU_CPUCpuUtil    274
6        OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc    272
7      OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT    215
8          OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount    215
9                  OSLinux-OSLinux_ZABBIX_Host_Uptime    215
10      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies    215
11       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT    213
12                            OSLinux-CPU_CPU_CPULoad    210
13  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...    210
14                        OSLinux-CPU_CPU_CPUUserTime    210
15                        OSLinux-CPU_CPU_CPUidleutil    210
16                             OSLinux-CPU_CPU_CPUWio    210
17                         OSLinux-CPU_CPU_CPUSysTime    210
18     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps    146
19  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRea...    146, 'min_timestamp_unix_and_local': (1616481000, '2021-03-23 14:30:00 CST'), 'max_timestamp_unix_and_local': (1616482800, '2021-03-23 15:00:00 CST'), 'sample_rows_up_to_5':     timestamp cmdb_id                                     kpi_name         value
0  1616481000    IG01     JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09
1  1616481000    IG01   JVM-Memory_7778_JVM_Memory_HeapMemoryUsage  4.306550e+01
2  1616481000    IG01    JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  3.226738e+08
3  1616481000    IG01  JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  5.867203e+07
4  1616481000    IG01    JVM-Operating System_7778_JVM_JVM_CPULoad  5.364000e-01}, {'row_count': 471, 'unique_cmdb_top20':     cmdb_id  count
0  Tomcat04     87
1  Tomcat02     86
2  Tomcat01     86
3  Tomcat03     86
4  apache01     62
5  apache02     62
6   Mysql02      2, 'unique_log_name_top20':            log_name  count
0   log.error_count    187
1     log.row_count    187
2  log.gc_pause_max     97, 'min_timestamp_unix_and_local': (1616481000, '2021-03-23 14:30:00 CST'), 'max_timestamp_unix_and_local': (1616482800, '2021-03-23 15:00:00 CST'), 'sample_rows_up_to_5':     timestamp   cmdb_id          log_name   value
0  1616481000  Tomcat01   log.error_count    0.00
1  1616481000  Tomcat01  log.gc_pause_max    0.06
2  1616481000  Tomcat01     log.row_count  519.00
3  1616481000  Tomcat02   log.error_count    0.00
4  1616481000  Tomcat02     log.row_count  487.00}, {'row_count': 264, 'unique_cmdb_top20':      cmdb_id  count
0       MG02     42
1       MG01     42
2       IG02     18
3       IG01     18
4   Tomcat01     18
5   Tomcat02     18
6   Tomcat03     18
7   Tomcat04     18
8   dockerA1     18
9   dockerA2     18
10  dockerB1     18
11  dockerB2     18, 'unique_trace_name_top20':                        trace_name  count
0   trace.from_root.duration_mean     12
1    trace.from_root.duration_p95     12
2       trace.from_root.row_count     12
3        trace.self.duration_mean     12
4         trace.self.duration_p95     12
5            trace.self.row_count     12
6      trace.to_MG02.duration_p95      8
7         trace.to_MG02.row_count      8
8      trace.to_MG01.duration_p95      8
9     trace.to_MG01.duration_mean      8
10    trace.to_MG02.duration_mean      8
11        trace.to_MG01.row_count      8
12      trace.from_MG02.row_count      4
13   trace.from_MG02.duration_p95      4
14      trace.from_IG01.row_count      4
15  trace.from_IG02.duration_mean      4
16  trace.from_IG01.duration_mean      4
17   trace.from_IG01.duration_p95      4
18  trace.from_MG01.duration_mean      4
19   trace.from_MG01.duration_p95      4, 'min_timestamp_unix_and_local': (1616460, '1970-01-20 01:01:00 CST'), 'max_timestamp_unix_and_local': (1616460, '1970-01-20 01:01:00 CST'), 'sample_rows_up_to_5':    timestamp cmdb_id                     trace_name       value
0    1616460    IG01  trace.from_root.duration_mean    0.480020
1    1616460    IG01   trace.from_root.duration_p95    2.081100
2    1616460    IG01      trace.from_root.row_count  102.000000
3    1616460    IG01       trace.self.duration_mean    0.523793
4    1616460    IG01        trace.self.duration_p95    2.125000}, {'row_count': 0, 'unique_cmdb_top20': Empty DataFrame
Columns: [cmdb_id, count]
Index: [], 'top_messages_top20': Empty DataFrame
Columns: [message, count]
Index: [], 'min_timestamp_unix_and_local': (None, None), 'max_timestamp_unix_and_local': (None, None), 'sample_rows_up_to_5': Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: []})```
```

In [ ]:
# In[2]:

import numpy as np
import pandas as pd
import pytz

# reuse df_metric and tz from previous session; if not present, load minimally
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')
tz = pytz.timezone('Asia/Shanghai')

# Allowed root-cause components
allowed = ['apache01','apache02','Tomcat01','Tomcat02','Tomcat03','Tomcat04',
           'MG01','MG02','IG01','IG02','Mysql01','Mysql02','Redis01','Redis02']

# Filter metric to allowed components
df_m = df_metric[df_metric['cmdb_id'].isin(allowed)].copy()

# Ensure numeric timestamp and value
df_m['timestamp'] = pd.to_numeric(df_m['timestamp'], errors='coerce')
df_m['value'] = pd.to_numeric(df_m['value'], errors='coerce')

# 1) Compute global P95 and P5 for each component-KPI using all available rows (global thresholds)
grp = df_m.groupby(['cmdb_id','kpi_name'])['value']
df_thresh = grp.quantile(0.95).rename('global_P95').to_frame()
df_thresh['global_P5'] = grp.quantile(0.05).values
df_thresh = df_thresh.reset_index()

# 2) Filter metric rows into the incident time window inclusive
start_ts = 1616481000
end_ts = 1616482800
df_window = df_m[(df_m['timestamp'] >= start_ts) & (df_m['timestamp'] <= end_ts)].copy()

# Merge thresholds into window rows to evaluate breaches
df_w = df_window.merge(df_thresh, on=['cmdb_id','kpi_name'], how='left')

# Mark points above global P95 (only treat anomalies as points > global_P95)
df_w['above_p95'] = df_w['value'] > df_w['global_P95']

# For earliest timestamp above P95, produce a ts_above column where non-above values are NaN
df_w['ts_above'] = np.where(df_w['above_p95'], df_w['timestamp'], np.nan)

# Aggregate window-level summaries per component-KPI
agg_window = df_w.groupby(['cmdb_id','kpi_name']).agg(
    total_points_in_window=('value','size'),
    count_points_above_P95=('above_p95','sum'),
    earliest_ts_above_unix=('ts_above','min'),
    max_value_in_window=('value','max')
).reset_index()

# Now combine with global thresholds to ensure we include series with zero points in window
df_summary = df_thresh.merge(agg_window, on=['cmdb_id','kpi_name'], how='left')

# Fill NaNs for series with no window points
df_summary['total_points_in_window'] = df_summary['total_points_in_window'].fillna(0).astype(int)
df_summary['count_points_above_P95'] = df_summary['count_points_above_P95'].fillna(0).astype(int)
df_summary['max_value_in_window'] = df_summary['max_value_in_window']  # keep NaN if no data
df_summary['earliest_ts_above_unix'] = df_summary['earliest_ts_above_unix']

# Convert earliest timestamp above to localized string or None
def format_ts(x):
    if pd.isna(x):
        return None
    try:
        dt = pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz)
        return dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    except Exception:
        return None

df_summary['earliest_timestamp_above_P95'] = df_summary['earliest_ts_above_unix'].apply(format_ts)

# Compute breach_ratio = (max_value_in_window - global_P95)/global_P95 ; null if global_P95 == 0 or max is NaN
def calc_breach(row):
    gp = row['global_P95']
    mv = row['max_value_in_window']
    if pd.isna(mv) or pd.isna(gp) or gp == 0:
        return None
    return (mv - gp) / gp

df_summary['breach_ratio'] = df_summary.apply(calc_breach, axis=1)

# Select and order columns for compact summary
df_summary_compact = df_summary[[
    'cmdb_id','kpi_name',
    'total_points_in_window','count_points_above_P95','earliest_timestamp_above_P95',
    'max_value_in_window','global_P95','breach_ratio'
]]

# Number of component-kpi series processed
n_series = df_summary_compact.shape[0]

# Top 20 by count_points_above_P95 descending
top20 = df_summary_compact.sort_values(
    by=['count_points_above_P95','breach_ratio'], ascending=[False, False]
).head(20).reset_index(drop=True)

# Display compact outputs
n_series, top20

```
Out[2]:
```
Summary (plain English)

- Number of component–KPI series processed (allowed components): 1,110.

- Definition/reminder: anomalies = points strictly greater than the global P95 computed from the entire series (global across time). The incident window used: timestamps >= 1616481000 and <= 1616482800. Timestamps shown are localized to CST (UTC+8). breach_ratio = (max_in_window - global_P95) / global_P95.

Top 20 component–KPI summaries (sorted by count_points_above_P95, compact):

1. Tomcat03, OSLinux-CPU_CPU_CPUWio — total_points_in_window 30, count_above_P95 2, earliest_above 2021-03-23 14:55:00 CST, max 2.468000, global_P95 0.490962, breach_ratio 4.026869  
2. Tomcat02, OSLinux-CPU_CPU_CPUWio — 30, 2, earliest 2021-03-23 14:42:00 CST, max 2.351815, P95 0.471163, breach_ratio 3.991507  
3. Tomcat03, OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT — 31, 2, earliest 2021-03-23 14:36:00 CST, max 1.000000, P95 0.500000, breach_ratio 1.000000  
4. MG01, OSLinux-CPU_CPU_CPUWio — 30, 2, earliest 2021-03-23 14:40:00 CST, max 1.214895, P95 0.683020, breach_ratio 0.778711  
5. Mysql01, Mysql-MySQL_3306_Max trx rows locked — 29, 2, earliest 2021-03-23 14:37:00 CST, max 1.000000, P95 0.600000, breach_ratio 0.666667  
6. Mysql01, Mysql-MySQL_3306_MaxTrxRowsModified — 29, 2, earliest 2021-03-23 14:37:00 CST, max 1.000000, P95 0.600000, breach_ratio 0.666667  
7. Mysql02, Mysql-MySQL_3306_LongestTrxActiveTime — 29, 2, earliest 2021-03-23 14:53:00 CST, max 1.000000, P95 0.600000, breach_ratio 0.666667  
8. Mysql01, Mysql-MySQL_3306_max trx lock memory bytes — 29, 2, earliest 2021-03-23 14:37:00 CST, max 1136.000000, P95 681.600000, breach_ratio 0.666667  
9. Redis02, OSLinux-CPU_CPU_CPUWio — 30, 2, earliest 2021-03-23 14:31:00 CST, max 0.105100, P95 0.065375, breach_ratio 0.607642  
10. MG02, OSLinux-CPU_CPU_CPUWio — 30, 2, earliest 2021-03-23 14:51:00 CST, max 0.665400, P95 0.425345, breach_ratio 0.564378  
11. Tomcat04, Tomcat-Threads_7441-"http-nio-8003"_CurrentThreadsBusy — 31, 2, earliest 2021-03-23 14:42:00 CST, max 10.000000, P95 8.000000, breach_ratio 0.250000  
12. Mysql02, Mysql-MySQL_3306_ThreadsRunning — 29, 2, earliest 2021-03-23 14:46:00 CST, max 15.000000, P95 12.400000, breach_ratio 0.209677  
13. Tomcat04, OSLinux-CPU_CPU_CPUWio — 30, 2, earliest 2021-03-23 14:41:00 CST, max 0.461500, P95 0.384044, breach_ratio 0.201684  
14. Redis02, OSLinux-CPU_CPU_CPULoad — 30, 2, earliest 2021-03-23 14:58:00 CST, max 0.320000, P95 0.269500, breach_ratio 0.187384  
15. Tomcat01, OSLinux-CPU_CPU_CPUWio — 30, 2, earliest 2021-03-23 14:45:00 CST, max 0.428453, P95 0.363960, breach_ratio 0.177197  
16. Tomcat03, Tomcat-Threads_7441-"http-nio-8003"_CurrentThreadsBusy — 31, 2, earliest 2021-03-23 14:57:00 CST, max 10.000000, P95 8.500000, breach_ratio 0.176471  
17. Mysql01, Mysql-MySQL_3306_Innodb buffer pool pages dirty — 29, 2, earliest 2021-03-23 14:33:00 CST, max 136.000000, P95 116.400000, breach_ratio 0.168385  
18. Tomcat02, OSLinux-CPU_CPU_CPULoad — 30, 2, earliest 2021-03-23 14:39:00 CST, max 1.670000, P95 1.460500, breach_ratio 0.143444  
19. Mysql02, Mysql-MySQL_3306_Innodb buffer pool pages misc — 29, 2, earliest 2021-03-23 14:54:00 CST, max 131.000000, P95 122.600000, breach_ratio 0.068515  
20. Tomcat04, OSLinux-CPU_CPU_CPULoad — 30, 2, earliest 2021-03-23 14:39:00 CST, max 1.380000, P95 1.295000, breach_ratio 0.065637

Takeaway: among the allowed components, many component–KPI series had a small number of points exceeding their global P95 during the incident window (top entries mostly show exactly 2 points above P95). The most extreme relative breaches (largest breach_ratio) in the top-20 are CPU WIO on Tomcat03/Tomcat02 (breach_ratio ~4.03 and ~3.99), indicating those maxima were several times their historical P95. Traceback or further root-cause analysis should focus on the component–KPIs with high breach_ratio and multiple exceedances (start with the top few listed).

The original code execution output of IPython Kernel is also provided below for reference:

(1110,      cmdb_id                                           kpi_name  total_points_in_window  count_points_above_P95 earliest_timestamp_above_P95  max_value_in_window  global_P95  breach_ratio
0   Tomcat03                             OSLinux-CPU_CPU_CPUWio                      30                       2      2021-03-23 14:55:00 CST             2.468000    0.490962      4.026869
1   Tomcat02                             OSLinux-CPU_CPU_CPUWio                      30                       2      2021-03-23 14:42:00 CST             2.351815    0.471163      3.991507
2   Tomcat03     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT                      31                       2      2021-03-23 14:36:00 CST             1.000000    0.500000      1.000000
3       MG01                             OSLinux-CPU_CPU_CPUWio                      30                       2      2021-03-23 14:40:00 CST             1.214895    0.683020      0.778711
4    Mysql01               Mysql-MySQL_3306_Max trx rows locked                      29                       2      2021-03-23 14:37:00 CST             1.000000    0.600000      0.666667
5    Mysql01                Mysql-MySQL_3306_MaxTrxRowsModified                      29                       2      2021-03-23 14:37:00 CST             1.000000    0.600000      0.666667
6    Mysql02              Mysql-MySQL_3306_LongestTrxActiveTime                      29                       2      2021-03-23 14:53:00 CST             1.000000    0.600000      0.666667
7    Mysql01         Mysql-MySQL_3306_max trx lock memory bytes                      29                       2      2021-03-23 14:37:00 CST          1136.000000  681.600000      0.666667
8    Redis02                             OSLinux-CPU_CPU_CPUWio                      30                       2      2021-03-23 14:31:00 CST             0.105100    0.065375      0.607642
9       MG02                             OSLinux-CPU_CPU_CPUWio                      30                       2      2021-03-23 14:51:00 CST             0.665400    0.425345      0.564378
10  Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...                      31                       2      2021-03-23 14:42:00 CST            10.000000    8.000000      0.250000
11   Mysql02                    Mysql-MySQL_3306_ThreadsRunning                      29                       2      2021-03-23 14:46:00 CST            15.000000   12.400000      0.209677
12  Tomcat04                             OSLinux-CPU_CPU_CPUWio                      30                       2      2021-03-23 14:41:00 CST             0.461500    0.384044      0.201684
13   Redis02                            OSLinux-CPU_CPU_CPULoad                      30                       2      2021-03-23 14:58:00 CST             0.320000    0.269500      0.187384
14  Tomcat01                             OSLinux-CPU_CPU_CPUWio                      30                       2      2021-03-23 14:45:00 CST             0.428453    0.363960      0.177197
15  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...                      31                       2      2021-03-23 14:57:00 CST            10.000000    8.500000      0.176471
16   Mysql01    Mysql-MySQL_3306_Innodb buffer pool pages dirty                      29                       2      2021-03-23 14:33:00 CST           136.000000  116.400000      0.168385
17  Tomcat02                            OSLinux-CPU_CPU_CPULoad                      30                       2      2021-03-23 14:39:00 CST             1.670000    1.460500      0.143444
18   Mysql02     Mysql-MySQL_3306_Innodb buffer pool pages misc                      29                       2      2021-03-23 14:54:00 CST           131.000000  122.600000      0.068515
19  Tomcat04                            OSLinux-CPU_CPU_CPULoad                      30                       2      2021-03-23 14:39:00 CST             1.380000    1.295000      0.065637)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

# reuse df_metric if present, else load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

tz = pytz.timezone('Asia/Shanghai')

# Ensure numeric types
df_metric['timestamp'] = pd.to_numeric(df_metric['timestamp'], errors='coerce')
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

# Incident window
start_ts = 1616481000
end_ts = 1616482800

# Target series list
targets = [
    ('Tomcat03','OSLinux-CPU_CPU_CPUWio'),
    ('Tomcat02','OSLinux-CPU_CPU_CPUWio'),
    ('Tomcat03','OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT'),
    ('MG01','OSLinux-CPU_CPU_CPUWio'),
    ('Mysql01','Mysql-MySQL_3306_Max trx rows locked'),
    ('Redis02','OSLinux-CPU_CPU_CPUWio'),
]

# Compute global P95 for these series using entire metric.csv (global thresholds)
df_glob = df_metric[df_metric[['cmdb_id','kpi_name']].apply(tuple, axis=1).isin(targets)].copy()
glob_p95 = df_glob.groupby(['cmdb_id','kpi_name'])['value'].quantile(0.95).rename('global_P95').reset_index()

# Make a mapping for quick lookup
p95_map = {(r['cmdb_id'], r['kpi_name']): r['global_P95'] for _, r in glob_p95.iterrows()}

# Filter window rows for targets
df_window = df_metric[
    (df_metric['timestamp'] >= start_ts) &
    (df_metric['timestamp'] <= end_ts) &
    (df_metric[['cmdb_id','kpi_name']].apply(tuple, axis=1).isin(targets))
].copy()

# Prepare function to localize timestamp to CST string
def to_cst_str(ts):
    if pd.isna(ts):
        return None
    try:
        return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    except Exception:
        return None

# For each target, build points table and identify consecutive anomaly segments
points_list = []
segments_list = []
summary_rows = []

for cmdb,kpi in targets:
    gp95 = p95_map.get((cmdb,kpi), np.nan)
    # all points in window for this series
    df_s = df_window[(df_window['cmdb_id']==cmdb) & (df_window['kpi_name']==kpi)].copy()
    df_s = df_s.sort_values('timestamp').reset_index(drop=True)
    # mark above P95 (strictly greater)
    df_s['is_above_P95'] = df_s['value'] > gp95
    # add localized timestamp for readability
    df_s['timestamp_local'] = df_s['timestamp'].apply(to_cst_str)
    # store points (compact): include series id columns
    if df_s.empty:
        pts = pd.DataFrame(columns=['cmdb_id','kpi_name','timestamp','timestamp_local','value','is_above_P95'])
    else:
        pts = df_s[['cmdb_id','kpi_name','timestamp','timestamp_local','value','is_above_P95']].copy()
    points_list.append(pts)
    # Identify consecutive anomaly segments
    df_anom = df_s[df_s['is_above_P95']].copy()
    if df_anom.empty:
        segs = pd.DataFrame(columns=[
            'cmdb_id','kpi_name','segment_start_timestamp','segment_end_timestamp',
            'segment_length_in_minutes','segment_max_value','segment_breach_ratio'
        ])
        num_segs = 0
        earliest_seg_start = None
        most_severe_ratio = None
    else:
        # compute diff in seconds between consecutive anomaly rows
        df_anom = df_anom.sort_values('timestamp').reset_index(drop=True)
        df_anom['diff'] = df_anom['timestamp'].diff().fillna(9999).astype(int)
        # new segment where diff != 60
        df_anom['seg_id'] = (df_anom['diff'] != 60).cumsum()
        agg = df_anom.groupby('seg_id').agg(
            segment_start_timestamp_unix=('timestamp','min'),
            segment_end_timestamp_unix=('timestamp','max'),
            segment_length_in_minutes=('timestamp','count'),
            segment_max_value=('value','max')
        ).reset_index(drop=True)
        # compute breach ratios and localized timestamps
        agg['segment_breach_ratio'] = agg['segment_max_value'].apply(
            lambda mv: None if (pd.isna(gp95) or gp95==0) else (mv - gp95)/gp95
        )
        agg['segment_start_timestamp'] = agg['segment_start_timestamp_unix'].apply(to_cst_str)
        agg['segment_end_timestamp'] = agg['segment_end_timestamp_unix'].apply(to_cst_str)
        # attach series identifiers
        agg.insert(0,'kpi_name',kpi)
        agg.insert(0,'cmdb_id',cmdb)
        segs = agg[[
            'cmdb_id','kpi_name','segment_start_timestamp','segment_end_timestamp',
            'segment_length_in_minutes','segment_max_value','segment_breach_ratio'
        ]].copy()
        num_segs = segs.shape[0]
        earliest_seg_start = segs['segment_start_timestamp'].min() if num_segs>0 else None
        # most severe = max breach ratio (ignore None)
        if segs['segment_breach_ratio'].dropna().empty:
            most_severe_ratio = None
        else:
            most_severe_ratio = segs['segment_breach_ratio'].max()
    segments_list.append(segs)
    # summary row
    summary_rows.append({
        'cmdb_id': cmdb,
        'kpi_name': kpi,
        'global_P95': gp95,
        'number_of_anomalous_segments': num_segs,
        'earliest_segment_start_timestamp': earliest_seg_start,
        'most_severe_segment_breach_ratio': most_severe_ratio
    })

# Combine points and segments into compact tables for display
points_combined = pd.concat(points_list, ignore_index=True) if points_list else pd.DataFrame()
segments_combined = pd.concat(segments_list, ignore_index=True) if segments_list else pd.DataFrame()
summary_df = pd.DataFrame(summary_rows)

# For clarity, order summary by most_severe_segment_breach_ratio desc then number_of_anomalous_segments desc
summary_df = summary_df.sort_values(by=['most_severe_segment_breach_ratio','number_of_anomalous_segments'],
                                    ascending=[False, False]).reset_index(drop=True)

# Keep outputs compact:
# - global P95 values per target: summary_df[['cmdb_id','kpi_name','global_P95']]
# - points_combined: show full window rows (should be ~30 per series)
# - segments_combined: anomaly segments
# - summary_df selected columns

summary_df[['cmdb_id','kpi_name','global_P95','number_of_anomalous_segments',
            'earliest_segment_start_timestamp','most_severe_segment_breach_ratio']], points_combined, segments_combined

```
Out[3]:
```
Summary for the six requested series (all results use the previously computed global P95; incident window = 2021-03-23 14:30:00 CST to 2021-03-23 15:00:00 CST).

1) Tomcat03 — OSLinux-CPU_CPU_CPUWio
- global_P95: 0.490962
- points in window: 30 (timestamps 2021-03-23 14:30 → 15:00), each point marked True/False for > global_P95 (full table returned during analysis; 2 points above P95).
- Anomalous segments (consecutive minute-adjacent points where >P95):
  - Segment A: 2021-03-23 14:55:00 → 14:55:00 (length 1 min), max_value = 0.491094, segment_breach_ratio = 0.000269
  - Segment B: 2021-03-23 14:58:00 → 14:58:00 (length 1 min), max_value = 2.468000, segment_breach_ratio = 4.026869
- Compact summary: number_of_anomalous_segments = 2; earliest_segment_start = 2021-03-23 14:55:00 CST; most_severe_segment_breach_ratio = 4.026869

2) Tomcat02 — OSLinux-CPU_CPU_CPUWio
- global_P95: 0.471163
- points in window: 30; 2 points above P95.
- Anomalous segments:
  - Segment A: 2021-03-23 14:42:00 → 14:42:00 (1 min), max_value = 2.351815, segment_breach_ratio = 3.991507
  - Segment B: 2021-03-23 15:00:00 → 15:00:00 (1 min), max_value = 0.536673, segment_breach_ratio = 0.139038
- Compact summary: number_of_anomalous_segments = 2; earliest_segment_start = 2021-03-23 14:42:00 CST; most_severe_segment_breach_ratio = 3.991507

3) Tomcat03 — OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT
- global_P95: 0.500000
- points in window: 31; 2 points above P95.
- Anomalous segments:
  - Segment A: 2021-03-23 14:36:00 → 14:36:00 (1 min), max_value = 1.000000, segment_breach_ratio = 1.000000
  - Segment B: 2021-03-23 14:41:00 → 14:41:00 (1 min), max_value = 1.000000, segment_breach_ratio = 1.000000
- Compact summary: number_of_anomalous_segments = 2; earliest_segment_start = 2021-03-23 14:36:00 CST; most_severe_segment_breach_ratio = 1.000000

4) MG01 — OSLinux-CPU_CPU_CPUWio
- global_P95: 0.683020
- points in window: 30; 2 points above P95.
- Anomalous segments:
  - Segment A: 2021-03-23 14:40:00 → 14:40:00 (1 min), max_value = 0.728200, segment_breach_ratio = 0.066147
  - Segment B: 2021-03-23 14:58:00 → 14:58:00 (1 min), max_value = 1.214895, segment_breach_ratio = 0.778711
- Compact summary: number_of_anomalous_segments = 2; earliest_segment_start = 2021-03-23 14:40:00 CST; most_severe_segment_breach_ratio = 0.778711

5) Mysql01 — Mysql-MySQL_3306_Max trx rows locked
- global_P95: 0.600000
- points in window: 29; 2 points above P95.
- Anomalous segments:
  - Segment A: 2021-03-23 14:37:00 → 14:37:00 (1 min), max_value = 1.000000, segment_breach_ratio = 0.666667
  - Segment B: 2021-03-23 14:59:00 → 14:59:00 (1 min), max_value = 1.000000, segment_breach_ratio = 0.666667
- Compact summary: number_of_anomalous_segments = 2; earliest_segment_start = 2021-03-23 14:37:00 CST; most_severe_segment_breach_ratio = 0.666667

6) Redis02 — OSLinux-CPU_CPU_CPUWio
- global_P95: 0.065375
- points in window: 30; 2 points above P95.
- Anomalous segments:
  - Segment A: 2021-03-23 14:31:00 → 14:31:00 (1 min), max_value = 0.067200, segment_breach_ratio = 0.027912
  - Segment B: 2021-03-23 14:36:00 → 14:36:00 (1 min), max_value = 0.105100, segment_breach_ratio = 0.607642
- Compact summary: number_of_anomalous_segments = 2; earliest_segment_start = 2021-03-23 14:31:00 CST; most_severe_segment_breach_ratio = 0.607642

Overall observations and guidance:
- Every series had exactly two anomaly points in the incident window, but those anomalies were isolated single-minute spikes (segments length = 1 minute each).
- The most severe relative breaches are:
  - Tomcat03 CPU WIO at 14:58 (breach_ratio ≈ 4.03)
  - Tomcat02 CPU WIO at 14:42 (breach_ratio ≈ 3.99)
  These are large multiples of the historical P95 and therefore highest priority for follow-up.
- Other notable breaches (moderate severity): Tomcat03 TCP CLOSE-WAIT (breach_ratio = 1.0), MG01 CPU WIO (0.78), Mysql01 locking metric (0.667), Redis02 (0.608).
- Most anomalies are single-minute spikes rather than sustained multi-minute runs. Investigate the high-breach single minutes first (Tomcat03 and Tomcat02 CPU WIO), then check Tomcat03 TCP CLOSE-WAIT and Mysql01 locking behavior around their spike times.

If you want, I can (for any specific series above) print the full per-minute rows within the window (timestamp, value, is_above_P95) to inspect exact values and neighbor context.

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id                                        kpi_name  global_P95  number_of_anomalous_segments earliest_segment_start_timestamp  most_severe_segment_breach_ratio
0  Tomcat03                          OSLinux-CPU_CPU_CPUWio    0.490962                             2          2021-03-23 14:55:00 CST                          4.026869
1  Tomcat02                          OSLinux-CPU_CPU_CPUWio    0.471163                             2          2021-03-23 14:42:00 CST                          3.991507
2  Tomcat03  OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT    0.500000                             2          2021-03-23 14:36:00 CST                          1.000000
3      MG01                          OSLinux-CPU_CPU_CPUWio    0.683020                             2          2021-03-23 14:40:00 CST                          0.778711
4   Mysql01            Mysql-MySQL_3306_Max trx rows locked    0.600000                             2          2021-03-23 14:37:00 CST                          0.666667
5   Redis02                          OSLinux-CPU_CPU_CPUWio    0.065375                             2          2021-03-23 14:31:00 CST                          0.607642,       cmdb_id                kpi_name   timestamp          timestamp_local     value  is_above_P95
0    Tomcat03  OSLinux-CPU_CPU_CPUWio  1616481000  2021-03-23 14:30:00 CST  0.353800         False
1    Tomcat03  OSLinux-CPU_CPU_CPUWio  1616481060  2021-03-23 14:31:00 CST  0.025000         False
2    Tomcat03  OSLinux-CPU_CPU_CPUWio  1616481120  2021-03-23 14:32:00 CST  0.062412         False
3    Tomcat03  OSLinux-CPU_CPU_CPUWio  1616481180  2021-03-23 14:33:00 CST  0.045800         False
4    Tomcat03  OSLinux-CPU_CPU_CPUWio  1616481240  2021-03-23 14:34:00 CST  0.129000         False
..        ...                     ...         ...                      ...       ...           ...
175   Redis02  OSLinux-CPU_CPU_CPUWio  1616482560  2021-03-23 14:56:00 CST  0.008400         False
176   Redis02  OSLinux-CPU_CPU_CPUWio  1616482620  2021-03-23 14:57:00 CST  0.004200         False
177   Redis02  OSLinux-CPU_CPU_CPUWio  1616482680  2021-03-23 14:58:00 CST  0.063145         False
178   Redis02  OSLinux-CPU_CPU_CPUWio  1616482740  2021-03-23 14:59:00 CST  0.004200         False
179   Redis02  OSLinux-CPU_CPU_CPUWio  1616482800  2021-03-23 15:00:00 CST  0.021000         False

[180 rows x 6 columns],      cmdb_id                                        kpi_name  segment_start_timestamp    segment_end_timestamp  segment_length_in_minutes  segment_max_value  segment_breach_ratio
0   Tomcat03                          OSLinux-CPU_CPU_CPUWio  2021-03-23 14:55:00 CST  2021-03-23 14:55:00 CST                          1           0.491094              0.000269
1   Tomcat03                          OSLinux-CPU_CPU_CPUWio  2021-03-23 14:58:00 CST  2021-03-23 14:58:00 CST                          1           2.468000              4.026869
2   Tomcat02                          OSLinux-CPU_CPU_CPUWio  2021-03-23 14:42:00 CST  2021-03-23 14:42:00 CST                          1           2.351815              3.991507
3   Tomcat02                          OSLinux-CPU_CPU_CPUWio  2021-03-23 15:00:00 CST  2021-03-23 15:00:00 CST                          1           0.536673              0.139038
4   Tomcat03  OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT  2021-03-23 14:36:00 CST  2021-03-23 14:36:00 CST                          1           1.000000              1.000000
5   Tomcat03  OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT  2021-03-23 14:41:00 CST  2021-03-23 14:41:00 CST                          1           1.000000              1.000000
6       MG01                          OSLinux-CPU_CPU_CPUWio  2021-03-23 14:40:00 CST  2021-03-23 14:40:00 CST                          1           0.728200              0.066147
7       MG01                          OSLinux-CPU_CPU_CPUWio  2021-03-23 14:58:00 CST  2021-03-23 14:58:00 CST                          1           1.214895              0.778711
8    Mysql01            Mysql-MySQL_3306_Max trx rows locked  2021-03-23 14:37:00 CST  2021-03-23 14:37:00 CST                          1           1.000000              0.666667
9    Mysql01            Mysql-MySQL_3306_Max trx rows locked  2021-03-23 14:59:00 CST  2021-03-23 14:59:00 CST                          1           1.000000              0.666667
10   Redis02                          OSLinux-CPU_CPU_CPUWio  2021-03-23 14:31:00 CST  2021-03-23 14:31:00 CST                          1           0.067200              0.027912
11   Redis02                          OSLinux-CPU_CPU_CPUWio  2021-03-23 14:36:00 CST  2021-03-23 14:36:00 CST                          1           0.105100              0.607642)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

# reuse df_log if present, else load
try:
    df_log
except NameError:
    df_log = pd.read_csv('log.csv')

tz = pytz.timezone('Asia/Shanghai')

# Ensure numeric timestamp and value
df_log['timestamp'] = pd.to_numeric(df_log['timestamp'], errors='coerce')
df_log['value'] = pd.to_numeric(df_log['value'], errors='coerce')

# Incident window
start_ts = 1616481000
end_ts = 1616482800

# Components of interest
components = ['Tomcat03','Tomcat02']

# Filter rows for components and window
df_log_win = df_log[
    df_log['cmdb_id'].isin(components) &
    (df_log['timestamp'] >= start_ts) &
    (df_log['timestamp'] <= end_ts)
].copy().sort_values(['cmdb_id','timestamp']).reset_index(drop=True)

# helper to localize timestamp
def to_cst_str(x):
    if pd.isna(x):
        return None
    return pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

df_log_win['timestamp_local'] = df_log_win['timestamp'].apply(to_cst_str)

# Prepare per-component reports
reports = {}
for comp in components:
    dfc = df_log_win[df_log_win['cmdb_id']==comp].copy()
    total_rows = dfc.shape[0]
    # counts per log_name (top entries)
    counts = dfc['log_name'].value_counts().rename_axis('log_name').reset_index(name='count').head(20)
    # error_count rows with value > 0 (up to 20)
    errs = dfc[(dfc['log_name']=='log.error_count') & (dfc['value']>0)].copy()
    errs = errs[['timestamp','timestamp_local','log_name','value']].head(20)
    # top 5 rows by value for log_name containing 'gc' or 'gc_pause' (case-insensitive)
    gc_mask = dfc['log_name'].str.contains('gc|gc_pause', case=False, na=False)
    gc_top5 = dfc[gc_mask].sort_values('value', ascending=False)[['timestamp','timestamp_local','log_name','value']].head(5)
    # rows within ±2 minutes (±120s) of metric anomaly timestamps
    anomaly_map = {'Tomcat03':1616482680, 'Tomcat02':1616481720}
    ts0 = anomaly_map.get(comp)
    near = pd.DataFrame(columns=dfc.columns)
    if ts0 is not None:
        near = dfc[(dfc['timestamp'] >= (ts0-120)) & (dfc['timestamp'] <= (ts0+120))].copy()
        near = near[['timestamp','timestamp_local','log_name','value']].sort_values('timestamp')
    # store compact report
    reports[comp] = {
        'total_rows_in_window': total_rows,
        'counts_per_log_name_top20': counts,
        'error_count_positive_rows_up_to_20': errs,
        'gc_related_top5_by_value': gc_top5,
        'rows_near_metric_anomaly_±2min': near
    }

# Create a small summary table for both components
summary_df = pd.DataFrame([
    {
        'cmdb_id': comp,
        'total_rows_in_window': reports[comp]['total_rows_in_window'],
        'distinct_log_names': int(reports[comp]['counts_per_log_name_top20']['log_name'].nunique()) \
                              if not reports[comp]['counts_per_log_name_top20'].empty else 0,
        'top_log_names_sample': ', '.join(reports[comp]['counts_per_log_name_top20']['log_name'].head(3).tolist())
    }
    for comp in components
])

# Assign outputs to variables for compact display
summary_df, \
reports['Tomcat03']['counts_per_log_name_top20'], reports['Tomcat03']['error_count_positive_rows_up_to_20'], \
reports['Tomcat03']['gc_related_top5_by_value'], reports['Tomcat03']['rows_near_metric_anomaly_±2min'], \
reports['Tomcat02']['counts_per_log_name_top20'], reports['Tomcat02']['error_count_positive_rows_up_to_20'], \
reports['Tomcat02']['gc_related_top5_by_value'], reports['Tomcat02']['rows_near_metric_anomaly_±2min']

```
Out[4]:
```
Summary (compact plain English):

Overall window: 2021-03-23 14:30:00 CST – 15:00:00 CST.

Tomcat03
- Total rows in window: 86
- Counts per log_name (top): log.error_count 31, log.row_count 31, log.gc_pause_max 24
- log.error_count > 0 rows: none (no positive error_count values found)
- Top 5 gc/gc_pause rows (timestamp, value):
  - 2021-03-23 14:37:00 CST — log.gc_pause_max 0.10
  - 2021-03-23 14:36:00 CST — log.gc_pause_max 0.09
  - 2021-03-23 14:50:00 CST — log.gc_pause_max 0.09
  - 2021-03-23 14:45:00 CST — log.gc_pause_max 0.09
  - 2021-03-23 14:56:00 CST — log.gc_pause_max 0.09
- Rows within ±2 minutes of the metric anomaly at 1616482680 (14:58:00 CST): present. At the anomaly minute 2021-03-23 14:58:00 CST the logs show:
  - log.error_count = 0.00
  - log.gc_pause_max = 0.06
  - log.row_count = 515.00
  (Nearby minutes also show gc_pause_max and row_count values but no error_count > 0.)

Tomcat02
- Total rows in window: 86
- Counts per log_name (top): log.error_count 31, log.row_count 31, log.gc_pause_max 24
- log.error_count > 0 rows: none (no positive error_count values found)
- Top 5 gc/gc_pause rows (timestamp, value):
  - 2021-03-23 14:39:00 CST — log.gc_pause_max 0.11
  - 2021-03-23 14:35:00 CST — log.gc_pause_max 0.10
  - 2021-03-23 14:32:00 CST — log.gc_pause_max 0.09
  - 2021-03-23 14:33:00 CST — log.gc_pause_max 0.08
  - 2021-03-23 14:43:00 CST — log.gc_pause_max 0.08
- Rows within ±2 minutes of the metric anomaly at 1616481720 (14:42:00 CST): present. At the anomaly minute 2021-03-23 14:42:00 CST the logs show:
  - log.error_count = 0.00
  - log.gc_pause_max = 0.07
  - log.row_count = 563.00
  (Nearby minutes likewise show gc_pause_max and row_count values but no error_count > 0.)

Key takeaways
- Both Tomcat02 and Tomcat03 have the same set of log types and no positive log.error_count entries in this incident window.
- GC pause values are present and show modest spikes (Tomcat02 peak 0.11 at 14:39; Tomcat03 peak 0.10 at 14:37). 
- At the exact metric-anomaly minutes (Tomcat03 14:58, Tomcat02 14:42) there are no log.error_count > 0 entries; gc_pause_max values at those minutes are small/modest (0.06 and 0.07 respectively). 

If you want, I can print the (timestamp, log_name, value) rows near each anomaly minute for closer inspection.

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id  total_rows_in_window  distinct_log_names                              top_log_names_sample
0  Tomcat03                    86                   3  log.error_count, log.row_count, log.gc_pause_max
1  Tomcat02                    86                   3  log.error_count, log.row_count, log.gc_pause_max,            log_name  count
0   log.error_count     31
1     log.row_count     31
2  log.gc_pause_max     24, Empty DataFrame
Columns: [timestamp, timestamp_local, log_name, value]
Index: [],       timestamp          timestamp_local          log_name  value
107  1616481420  2021-03-23 14:37:00 CST  log.gc_pause_max   0.10
104  1616481360  2021-03-23 14:36:00 CST  log.gc_pause_max   0.09
143  1616482200  2021-03-23 14:50:00 CST  log.gc_pause_max   0.09
129  1616481900  2021-03-23 14:45:00 CST  log.gc_pause_max   0.09
160  1616482560  2021-03-23 14:56:00 CST  log.gc_pause_max   0.09,       timestamp          timestamp_local          log_name   value
159  1616482560  2021-03-23 14:56:00 CST   log.error_count    0.00
160  1616482560  2021-03-23 14:56:00 CST  log.gc_pause_max    0.09
161  1616482560  2021-03-23 14:56:00 CST     log.row_count  485.00
162  1616482620  2021-03-23 14:57:00 CST   log.error_count    0.00
163  1616482620  2021-03-23 14:57:00 CST     log.row_count  569.00
164  1616482680  2021-03-23 14:58:00 CST   log.error_count    0.00
165  1616482680  2021-03-23 14:58:00 CST  log.gc_pause_max    0.06
166  1616482680  2021-03-23 14:58:00 CST     log.row_count  515.00
167  1616482740  2021-03-23 14:59:00 CST   log.error_count    0.00
168  1616482740  2021-03-23 14:59:00 CST  log.gc_pause_max    0.06
169  1616482740  2021-03-23 14:59:00 CST     log.row_count  543.00
170  1616482800  2021-03-23 15:00:00 CST   log.error_count    0.00
171  1616482800  2021-03-23 15:00:00 CST     log.row_count   10.00,            log_name  count
0   log.error_count     31
1     log.row_count     31
2  log.gc_pause_max     24, Empty DataFrame
Columns: [timestamp, timestamp_local, log_name, value]
Index: [],      timestamp          timestamp_local          log_name  value
26  1616481540  2021-03-23 14:39:00 CST  log.gc_pause_max   0.11
15  1616481300  2021-03-23 14:35:00 CST  log.gc_pause_max   0.10
6   1616481120  2021-03-23 14:32:00 CST  log.gc_pause_max   0.09
9   1616481180  2021-03-23 14:33:00 CST  log.gc_pause_max   0.08
37  1616481780  2021-03-23 14:43:00 CST  log.gc_pause_max   0.08,      timestamp          timestamp_local          log_name   value
28  1616481600  2021-03-23 14:40:00 CST   log.error_count    0.00
29  1616481600  2021-03-23 14:40:00 CST     log.row_count  503.00
30  1616481660  2021-03-23 14:41:00 CST   log.error_count    0.00
31  1616481660  2021-03-23 14:41:00 CST  log.gc_pause_max    0.07
32  1616481660  2021-03-23 14:41:00 CST     log.row_count  556.00
33  1616481720  2021-03-23 14:42:00 CST   log.error_count    0.00
34  1616481720  2021-03-23 14:42:00 CST  log.gc_pause_max    0.07
35  1616481720  2021-03-23 14:42:00 CST     log.row_count  563.00
36  1616481780  2021-03-23 14:43:00 CST   log.error_count    0.00
37  1616481780  2021-03-23 14:43:00 CST  log.gc_pause_max    0.08
38  1616481780  2021-03-23 14:43:00 CST     log.row_count  509.00
39  1616481840  2021-03-23 14:44:00 CST   log.error_count    0.00
40  1616481840  2021-03-23 14:44:00 CST  log.gc_pause_max    0.03
41  1616481840  2021-03-23 14:44:00 CST     log.row_count  502.00)```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

# reuse df_trace if available, otherwise load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')

tz = pytz.timezone('Asia/Shanghai')

# Ensure numeric timestamp
df_trace['timestamp'] = pd.to_numeric(df_trace['timestamp'], errors='coerce')

# 1) Distinct trace timestamp values (up to 20) and count of distinct timestamps
unique_ts = pd.Series(np.sort(df_trace['timestamp'].dropna().unique()))
unique_count = unique_ts.shape[0]
unique_preview = unique_ts.head(20).reset_index(drop=True).to_frame(name='trace_timestamp')

# 2) Min and max trace.timestamp as-is
if unique_ts.empty:
    min_ts = None
    max_ts = None
else:
    min_ts = int(unique_ts.min())
    max_ts = int(unique_ts.max())

min_max_df = pd.DataFrame([{'min_timestamp_as_is': min_ts, 'max_timestamp_as_is': max_ts}])

# 3) Test scalings
multipliers = [1, 60, 1000, 1000000]
start_ts = 1616481000
end_ts = 1616482800

scaling_rows = []
samples_list = []

for m in multipliers:
    scaled = (unique_ts * m).astype('Int64')  # keep integer dtype, allow NA
    in_window_mask = (scaled >= start_ts) & (scaled <= end_ts)
    count_in_window = int(in_window_mask.sum())
    scaling_rows.append({'multiplier': m, 'count_in_window': count_in_window})
    if count_in_window > 0:
        matched = pd.DataFrame({
            'original_trace_timestamp': unique_ts[in_window_mask].astype('Int64').values,
            'scaled_timestamp': scaled[in_window_mask].astype('Int64').values
        })
        # keep up to 10 samples per multiplier
        matched_sample = matched.head(10).copy()
        matched_sample.insert(0, 'multiplier', m)
        samples_list.append(matched_sample)

scaling_summary = pd.DataFrame(scaling_rows)

# Combine sample rows (if any)
samples_df = pd.concat(samples_list, ignore_index=True) if samples_list else pd.DataFrame(
    columns=['multiplier','original_trace_timestamp','scaled_timestamp']
)

# For readability, also show scaled_timestamp localized datetime where applicable
def safe_local(ts):
    try:
        return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    except Exception:
        return None

if not samples_df.empty:
    samples_df['scaled_timestamp_local'] = samples_df['scaled_timestamp'].apply(safe_local)

# Compact outputs
unique_preview, pd.Series({'distinct_timestamps_count': unique_count}), min_max_df, scaling_summary, samples_df.head(40)

```
Out[5]:
```
from datetime import datetime
import pytz

# Prepare summary string based on previous analysis
tz = pytz.timezone('Asia/Shanghai')

orig_ts = 1616460
orig_local = datetime.utcfromtimestamp(orig_ts).replace(tzinfo=pytz.utc).astimezone(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

summary = (
    "Trace timestamp validation and rescaling check:\n\n"
    "1) Distinct trace.timestamp values (up to 20): [1616460]\n"
    "   - Count of distinct timestamps: 1\n\n"
    "2) Min and Max trace.timestamp as-is:\n"
    "   - min = 1616460, max = 1616460\n"
    f"   - Interpreted as UTC+8 local time: {orig_local} (this is 1970-01-20 01:01:00 CST)\n\n"
    "3) Scaling tests (how many scaled timestamps fall in the incident window "
    "[1616481000, 1616482800]):\n"
    "   - multiplier 1  -> count_in_window = 0\n"
    "   - multiplier 60 -> count_in_window = 0\n"
    "   - multiplier 1000 -> count_in_window = 0\n"
    "   - multiplier 1000000 -> count_in_window = 0\n\n"
    "   No scaled timestamps fall into the incident window for the tested multipliers, "
    "so trace timestamps do not align with metric/log epoch times under these common rescalings.\n\n"
    "Conclusion / recommendation:\n"
    "- The trace timestamps appear anomalous (a single small value ~1.6 million seconds). "
    "They do not match the metric/log timestamps (~1.616e9) under multipliers 1, 60, 1,000, or 1,000,000.\n"
    "- Investigate the trace ingestion/format: check whether timestamps were recorded in the wrong unit "
    "(e.g., minutes vs seconds vs milliseconds) or truncated/shifted, and confirm the source clock/timezone.\n"
    "- Once the correct scale/offset is identified, re-scale trace timestamps to align with metrics/logs for correlation."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(   trace_timestamp
0          1616460, distinct_timestamps_count    1
dtype: int64,    min_timestamp_as_is  max_timestamp_as_is
0              1616460              1616460,    multiplier  count_in_window
0           1                0
1          60                0
2        1000                0
3     1000000                0, Empty DataFrame
Columns: [multiplier, original_trace_timestamp, scaled_timestamp]
Index: [])```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz

# reuse df_metric if present, else load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

tz = pytz.timezone('Asia/Shanghai')

# Parameters
allowed = ['apache01','apache02','Tomcat01','Tomcat02','Tomcat03','Tomcat04',
           'MG01','MG02','IG01','IG02','Mysql01','Mysql02','Redis01','Redis02']
kpis = ['app.sr','app.rr','app.mrt','app.cnt']
start_ts = 1616481000
end_ts = 1616482800

# Prepare data: filter by allowed components and desired KPIs
df = df_metric[
    df_metric['cmdb_id'].isin(allowed) &
    df_metric['kpi_name'].isin(kpis)
].copy()

# Ensure numeric types
df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce')
df['value'] = pd.to_numeric(df['value'], errors='coerce')

# 1) Compute global P5 per component-KPI using entire available series (global)
p5_df = df.groupby(['cmdb_id','kpi_name'])['value'].quantile(0.05).reset_index().rename(columns={'value':'global_P5'})

# 2) Filter points into the incident window
df_win = df[(df['timestamp'] >= start_ts) & (df['timestamp'] <= end_ts)].copy()

# Merge global P5 into window rows for comparison
df_win = df_win.merge(p5_df, on=['cmdb_id','kpi_name'], how='left')

# 3) Mark points <= global_P5
df_win['below_p5'] = df_win['value'] <= df_win['global_P95'] if 'global_P95' in df_win.columns else df_win['value'] <= df_win['global_P5']

# (defensive: ensure using global_P5 column)
if 'below_p5' not in df_win.columns or df_win['below_p5'].isnull().all():
    df_win['below_p5'] = df_win['value'] <= df_win['global_P5']

# Aggregate per series
agg = df_win.groupby(['cmdb_id','kpi_name']).agg(
    total_points_in_window=('value','size'),
    count_points_below_P5=('below_p5','sum'),
    earliest_below_unix=('timestamp', lambda s: s[df_win.loc[s.index,'below_p5']].min() if not s.empty else np.nan),
    min_value_in_window=('value','min')
).reset_index()

# Merge global_P5 into agg (to include any series that might have window rows but missing p5)
agg = agg.merge(p5_df, on=['cmdb_id','kpi_name'], how='left')

# Format earliest timestamp to localized CST string (or None)
def fmt_ts(x):
    if pd.isna(x):
        return None
    try:
        return pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    except Exception:
        return None

agg['earliest_below_timestamp_local'] = agg['earliest_below_unix'].apply(fmt_ts)

# Keep only series with at least one point <= global_P5
agg_filtered = agg[agg['count_points_below_P5'] > 0].copy()

# Select and order columns compactly
result = agg_filtered[[
    'cmdb_id','kpi_name','global_P5','total_points_in_window',
    'count_points_below_P5','earliest_below_timestamp_local','min_value_in_window'
]]

# Sort by count_points_below_P5 desc, then min_value_in_window asc (more severe lower mins)
result = result.sort_values(by=['count_points_below_P5','min_value_in_window'], ascending=[False, True]).reset_index(drop=True)

# Number of series scanned (with these kpis across allowed components)
number_of_series_processed = p5_df.shape[0]

# Return compact output: number processed and top 20 rows
number_of_series_processed, result.head(20)

```
Out[6]:
```
Result summary (plain English):

- Number of component–KPI series processed: 0.
- Explanation: There are no metric series in metric.csv that match both (a) the requested KPI names ['app.sr','app.rr','app.mrt','app.cnt'] and (b) the allowed component list within the dataset. Therefore no global P5 could be computed and no window points meet the <= P5 condition.
- Next steps you can take:
  - Confirm the exact KPI names (case and punctuation) — these KPI names may not exist in metric.csv.
  - Or broaden the component list or KPI set, then re-run the analysis.
  - If you want, I can list available KPI names and their top components so you can pick the correct ones.

The original code execution output of IPython Kernel is also provided below for reference:

(0, Empty DataFrame
Columns: [cmdb_id, kpi_name, global_P5, total_points_in_window, count_points_below_P5, earliest_below_timestamp_local, min_value_in_window]
Index: [])```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz

# reuse existing df_metric and tz if available
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

tz = pytz.timezone('Asia/Shanghai')

# Parameters
components = ['Tomcat03','Tomcat02','MG01']
substrs = ['localdisk','dsk','read','write','io','iowait']
start_ts = 1616481000
end_ts = 1616482800

# Ensure numeric types
df_metric['timestamp'] = pd.to_numeric(df_metric['timestamp'], errors='coerce')
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

# Filter to the components of interest first (to reduce work)
df_cand = df_metric[df_metric['cmdb_id'].isin(components)].copy()

# Build case-insensitive mask for KPI name substrings
pattern = '|'.join([s for s in substrs])  # simple OR pattern
mask_kpi = df_cand['kpi_name'].str.contains(pattern, case=False, na=False)

df_matched = df_cand[mask_kpi].copy()

# Compute global P95 per component-kpi using full available series (must be BEFORE window filter)
group = df_matched.groupby(['cmdb_id','kpi_name'])['value']
df_global = group.quantile(0.95).reset_index().rename(columns={'value':'global_P95'})

# Filter window rows
df_window = df_matched[(df_matched['timestamp'] >= start_ts) & (df_matched['timestamp'] <= end_ts)].copy()

# Merge global thresholds into window rows
df_w = df_window.merge(df_global, on=['cmdb_id','kpi_name'], how='left')

# Mark points above global P95 (strict >)
df_w['above_p95'] = df_w['value'] > df_w['global_P95']

# Compute per-series aggregates in the window
agg = df_w.groupby(['cmdb_id','kpi_name']).agg(
    total_points_in_window=('value','size'),
    count_points_above_P95=('above_p95','sum'),
    earliest_ts_above_unix=('timestamp', lambda s: s[df_w.loc[s.index,'above_p95']].min() if not s.empty else np.nan),
    max_value_in_window=('value','max')
).reset_index()

# Merge with global P95 to ensure presence
agg = agg.merge(df_global, on=['cmdb_id','kpi_name'], how='left')

# Keep only series with at least one point > global_P95 in the window
agg_pos = agg[agg['count_points_above_P95'] > 0].copy()

# Format earliest timestamp to localized CST string or None
def to_cst(ts):
    if pd.isna(ts):
        return None
    try:
        return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    except Exception:
        return None

agg_pos['earliest_timestamp_above_P95'] = agg_pos['earliest_ts_above_unix'].apply(to_cst)

# Compute breach_ratio = (max - global_P95) / global_P95 ; null if global_P95 == 0 or NaN
def breach(row):
    gp = row['global_P95']
    mv = row['max_value_in_window']
    if pd.isna(gp) or pd.isna(mv) or gp == 0:
        return None
    return (mv - gp) / gp

agg_pos['breach_ratio'] = agg_pos.apply(breach, axis=1)

# Select and order compact columns
result = agg_pos[[
    'cmdb_id','kpi_name','total_points_in_window','count_points_above_P95',
    'earliest_timestamp_above_P95','max_value_in_window','global_P95','breach_ratio'
]].copy()

# Sort as requested and limit to top 20
result = result.sort_values(by=['count_points_above_P95','breach_ratio'], ascending=[False, False]).head(20).reset_index(drop=True)

# Number of series scanned (unique component-kpi combinations that matched substrings)
number_of_series_processed = df_global.shape[0]

# Display compact outputs
number_of_series_processed, result

```
Out[7]:
```
Summary of scan for KPI names containing any of ['localdisk','dsk','read','write','io','iowait'] for components Tomcat03, Tomcat02, MG01:

- Number of matching component–KPI series processed: 96.

Top results (series that had ≥1 point above their global P95 in the incident window; sorted by count_above_P95 then breach_ratio). For each: component, kpi_name, points_in_window, points_above_P95, earliest_above (CST), max_in_window, global_P95, breach_ratio.

1. Tomcat03 — OSLinux-CPU_CPU_CPUWio; 30 pts; 2 above; earliest 2021-03-23 14:55:00 CST; max 2.468; global_P95 0.490962; breach_ratio 4.0269  
2. Tomcat02 — OSLinux-CPU_CPU_CPUWio; 30; 2; earliest 2021-03-23 14:42:00 CST; max 2.351815; global_P95 0.471163; breach_ratio 3.9915  
3. MG01 — OSLinux-CPU_CPU_CPUWio; 30; 2; earliest 2021-03-23 14:40:00 CST; max 1.214895; global_P95 0.683020; breach_ratio 0.7787  
4. Tomcat03 — Tomcat-Threads_7441-"http-nio-8003"_CurrentThreadsBusy; 31; 2; earliest 2021-03-23 14:57:00 CST; max 10.0; global_P95 8.5; breach_ratio 0.1765  
5. MG01 — JVM-Threads_7779_JVM_ThreadCount_Threads; 31; 2; earliest 2021-03-23 14:45:00 CST; max 77.0; global_P95 76.5; breach_ratio 0.0065  
6. Tomcat03 — Tomcat-Requests_7441-"http-nio-8003"_RequestCount; 31; 2; earliest 2021-03-23 14:59:00 CST; max 2.740236e6; global_P95 2.739478e6; breach_ratio 0.000277  
7. Tomcat02 — Tomcat-Requests_7441-"http-nio-8003"_RequestCount; 31; 2; earliest 2021-03-23 14:59:00 CST; max 2.750429e6; global_P95 2.749690e6; breach_ratio 0.000269  
8. Tomcat03 — Tomcat-Requests_7441-"http-nio-8003"_ProcessingTime (or similar); 31; 2; earliest 2021-03-23 14:59:00 CST; max ~1.668247e9; global_P95 ~1.667845e9; breach_ratio 0.000241  
9. Tomcat02 — Tomcat-Requests_7441-"http-nio-8003"_ProcessingTime; 31; 2; earliest 2021-03-23 14:59:00 CST; max ~1.678829e9; global_P95 ~1.678429e9; breach_ratio 0.000238  
10. Tomcat02 — OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPerf...; 16; 1; earliest 2021-03-23 14:58:00 CST; max 0.3; global_P95 0.075; breach_ratio 3.0  
11. Tomcat03 — OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps; 16; 1; earliest 2021-03-23 15:00:00 CST; max 0.0167; global_P95 0.004175; breach_ratio 3.0  
12. Tomcat03 — OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead; 16; 1; earliest 2021-03-23 15:00:00 CST; max 0.1333; global_P95 0.033325; breach_ratio 3.0  
13. Tomcat03 — OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead; 16; 1; earliest 2021-03-23 14:58:00 CST; max 582.2667; global_P95 150.0667; breach_ratio 2.8801  
14. Tomcat03 — OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps; 16; 1; earliest 2021-03-23 14:58:00 CST; max 10.8833; global_P95 2.8583; breach_ratio 2.8076  
15. MG01 — OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead; 15; 1; earliest 2021-03-23 14:50:00 CST; max 106.5333; global_P95 36.15999; breach_ratio 1.9462  
16. Tomcat03 — OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps; 14; 1; earliest 2021-03-23 14:59:00 CST; max 11.0; global_P95 3.85; breach_ratio 1.8571  
17. Tomcat03 — OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead (another related metric entry); 14; 1; earliest 2021-03-23 14:59:00 CST; max 595.0; global_P95 217.35; breach_ratio 1.7375  
18. MG01 — OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps; 15; 1; earliest 2021-03-23 14:36:00 CST; max 6.0; global_P95 2.5; breach_ratio 1.4  
19. Tomcat03 — OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPerf...; 15; 1; earliest 2021-03-23 14:32:00 CST; max 0.7; global_P95 0.35; breach_ratio 1.0  
20. MG01 — OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps; 15; 1; earliest 2021-03-23 14:50:00 CST; max 0.5167; global_P95 0.28332; breach_ratio 0.8237

Takeaways / guidance:
- 96 matching series were evaluated; many showed a single window exceedance (count_points_above_P95 = 1) rather than sustained runs.  
- Notable disk/io-related anomalies (largest breach ratios and/or large absolute spikes) occur on Tomcat03 around 14:58–15:00 CST:
  - sda DSKRead and related sda DSKRTps/DSKTps show large relative breaches (breach_ratio ~2.8–2.88) and large absolute read values (hundreds of units).  
  - Tomcat02 also shows a sda DSKPerf spike at 14:58 with breach_ratio 3.0.  
- CPU WIO ("CPUWio") on Tomcat03 and Tomcat02 are top relative breaches (breach_ratio ≈ 4.03 and 3.99) — these matched because the substring 'io' matched CPUWio; these are high-priority signals to investigate together with the disk read spikes.  
- MG01 also shows disk read/bps spikes and a CPUWio breach (moderate severity).

Recommended next steps:
- Investigate Tomcat03 around 14:58–15:00 CST (sda read/rtps/tps spikes) and Tomcat02 at 14:58 (sda perf) together with the CPU WIO spikes on both Tomcat nodes — these co-located I/O and CPU WIO anomalies suggest transient disk I/O pressure possibly impacting Tomcat processes.  
- Examine host-level I/O wait, disk queue, and process-level I/O at those minutes; correlate with logs/traces if rescaled/realigned.

The original code execution output of IPython Kernel is also provided below for reference:

(96,      cmdb_id                                           kpi_name  total_points_in_window  count_points_above_P95 earliest_timestamp_above_P95  max_value_in_window    global_P95  breach_ratio
0   Tomcat03                             OSLinux-CPU_CPU_CPUWio                      30                       2      2021-03-23 14:55:00 CST         2.468000e+00  4.909617e-01      4.026869
1   Tomcat02                             OSLinux-CPU_CPU_CPUWio                      30                       2      2021-03-23 14:42:00 CST         2.351815e+00  4.711633e-01      3.991507
2       MG01                             OSLinux-CPU_CPU_CPUWio                      30                       2      2021-03-23 14:40:00 CST         1.214895e+00  6.830200e-01      0.778711
3   Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...                      31                       2      2021-03-23 14:57:00 CST         1.000000e+01  8.500000e+00      0.176471
4       MG01           JVM-Threads_7779_JVM_ThreadCount_Threads                      31                       2      2021-03-23 14:45:00 CST         7.700000e+01  7.650000e+01      0.006536
5   Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...                      31                       2      2021-03-23 14:59:00 CST         2.740236e+06  2.739478e+06      0.000277
6   Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...                      31                       2      2021-03-23 14:59:00 CST         2.750429e+06  2.749690e+06      0.000269
7   Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_Processin...                      31                       2      2021-03-23 14:59:00 CST         1.668247e+09  1.667845e+09      0.000241
8   Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_Processin...                      31                       2      2021-03-23 14:59:00 CST         1.678829e+09  1.678429e+09      0.000238
9   Tomcat02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...                      16                       1      2021-03-23 14:58:00 CST         3.000000e-01  7.500000e-02      3.000000
10  Tomcat03    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps                      16                       1      2021-03-23 15:00:00 CST         1.670000e-02  4.175000e-03      3.000000
11  Tomcat03    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead                      16                       1      2021-03-23 15:00:00 CST         1.333000e-01  3.332500e-02      3.000000
12  Tomcat03    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead                      16                       1      2021-03-23 14:58:00 CST         5.822667e+02  1.500667e+02      2.880053
13  Tomcat03    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps                      16                       1      2021-03-23 14:58:00 CST         1.088330e+01  2.858300e+00      2.807613
14      MG01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead                      15                       1      2021-03-23 14:50:00 CST         1.065333e+02  3.615999e+01      1.946165
15  Tomcat03     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps                      14                       1      2021-03-23 14:59:00 CST         1.100000e+01  3.850000e+00      1.857143
16  Tomcat03  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRea...                      14                       1      2021-03-23 14:59:00 CST         5.950000e+02  2.173500e+02      1.737520
17      MG01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps                      15                       1      2021-03-23 14:36:00 CST         6.000000e+00  2.500000e+00      1.400000
18  Tomcat03  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...                      15                       1      2021-03-23 14:32:00 CST         7.000000e-01  3.500000e-01      1.000000
19      MG01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps                      15                       1      2021-03-23 14:50:00 CST         5.167000e-01  2.833200e-01      0.823733)```
```